# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
if tf.config.list_physical_devices('GPU'):
    device = '/device:GPU:0'
else:
    device = '/cpu:0'

print('Using device: ', device)

class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

Using device:  /cpu:0
(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(channel_1, (5, 5), 
                                            padding='same', 
                                            activation='relu',
                                            kernel_initializer=initializer)

        self.conv2 = tf.keras.layers.Conv2D(channel_2, (3, 3), 
                                            padding='same', 
                                            activation='relu',
                                            kernel_initializer=initializer)

        self.flatten = tf.keras.layers.Flatten()

        self.fc = tf.keras.layers.Dense(num_classes, 
                                        activation='softmax',
                                        kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [7]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [10]:
print_every = 100

hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.2728307247161865, Accuracy: 9.375, Val Loss: 2.957157611846924, Val Accuracy: 11.300000190734863
Iteration 100, Epoch 1, Loss: 2.2469518184661865, Accuracy: 28.20235252380371, Val Loss: 1.9201295375823975, Val Accuracy: 36.099998474121094
Iteration 200, Epoch 1, Loss: 2.077571392059326, Accuracy: 31.91075897216797, Val Loss: 1.8740408420562744, Val Accuracy: 38.20000076293945
Iteration 300, Epoch 1, Loss: 1.9998064041137695, Accuracy: 33.86627960205078, Val Loss: 1.87184739112854, Val Accuracy: 37.400001525878906
Iteration 400, Epoch 1, Loss: 1.9328664541244507, Accuracy: 35.67643356323242, Val Loss: 1.7231765985488892, Val Accuracy: 42.69999694824219
Iteration 500, Epoch 1, Loss: 1.8899734020233154, Accuracy: 36.732784271240234, Val Loss: 1.6647216081619263, Val Accuracy: 43.20000076293945
Iteration 600, Epoch 1, Loss: 1.8576101064682007, Accuracy: 37.70798873901367, Val Loss: 1.6976633071899414, Val Accuracy: 41.70000076293945
Iteration 700, Epoch 1, Los

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [11]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, 
                                        momentum=0.9, 
                                        nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

print_every = 100
train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.1620984077453613, Accuracy: 9.375, Val Loss: 8.08535099029541, Val Accuracy: 8.0
Iteration 100, Epoch 1, Loss: 2.251762866973877, Accuracy: 23.468441009521484, Val Loss: 1.8712667226791382, Val Accuracy: 33.89999771118164
Iteration 200, Epoch 1, Loss: 2.0247528553009033, Accuracy: 29.780784606933594, Val Loss: 1.6759402751922607, Val Accuracy: 41.5
Iteration 300, Epoch 1, Loss: 1.9008312225341797, Accuracy: 33.814369201660156, Val Loss: 1.594890832901001, Val Accuracy: 43.599998474121094
Iteration 400, Epoch 1, Loss: 1.8041523694992065, Accuracy: 37.07138442993164, Val Loss: 1.5006310939788818, Val Accuracy: 46.29999923706055
Iteration 500, Epoch 1, Loss: 1.7356921434402466, Accuracy: 39.27457809448242, Val Loss: 1.4510533809661865, Val Accuracy: 47.099998474121094
Iteration 600, Epoch 1, Loss: 1.6890199184417725, Accuracy: 40.78099060058594, Val Loss: 1.4307441711425781, Val Accuracy: 48.69999694824219
Iteration 700, Epoch 1, Loss: 1.6485776901245117, Acc

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [14]:
learning_rate = 1e-2

def model_init_fn():
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(filters=32, kernel_size=(5, 5), padding='same', 
                               activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Conv2D(filters=16, kernel_size=(3, 3), padding='same', 
                               activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)
    ])
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.5754830837249756, Accuracy: 12.5, Val Loss: 4.844592094421387, Val Accuracy: 12.300000190734863
Iteration 100, Epoch 1, Loss: 1.995489239692688, Accuracy: 29.811264038085938, Val Loss: 1.7376799583435059, Val Accuracy: 38.29999923706055
Iteration 200, Epoch 1, Loss: 1.845713496208191, Accuracy: 34.96579360961914, Val Loss: 1.6178406476974487, Val Accuracy: 43.900001525878906
Iteration 300, Epoch 1, Loss: 1.7633365392684937, Accuracy: 37.82703399658203, Val Loss: 1.6215108633041382, Val Accuracy: 45.20000076293945
Iteration 400, Epoch 1, Loss: 1.6934385299682617, Accuracy: 40.173004150390625, Val Loss: 1.464455008506775, Val Accuracy: 47.29999923706055
Iteration 500, Epoch 1, Loss: 1.6447229385375977, Accuracy: 41.856910705566406, Val Loss: 1.434221625328064, Val Accuracy: 50.0
Iteration 600, Epoch 1, Loss: 1.6101751327514648, Accuracy: 43.02984619140625, Val Loss: 1.4091795682907104, Val Accuracy: 50.400001525878906
Iteration 700, Epoch 1, Loss: 1.57939064

Альтернативный менее гибкий способ обучения:

In [13]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 17s 21ms/step - loss: 1.8220 - sparse_categorical_accuracy: 0.3906 - val_loss: 1.7981 - val_sparse_categorical_accuracy: 0.3940
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.7786 - sparse_categorical_accuracy: 0.4082


[1.7786024808883667, 0.4081999957561493]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [17]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(32, (5, 5), padding='same', 
                               activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Conv2D(16, (3, 3), padding='same', 
                               activation='relu', kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 8e-4
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, 
                                        momentum=0.9, 
                                        nesterov=True)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.3684988021850586, Accuracy: 6.25, Val Loss: 2.6255688667297363, Val Accuracy: 10.199999809265137
Iteration 100, Epoch 1, Loss: 1.9304777383804321, Accuracy: 32.564971923828125, Val Loss: 1.706647276878357, Val Accuracy: 40.599998474121094
Iteration 200, Epoch 1, Loss: 1.8030812740325928, Accuracy: 36.76927947998047, Val Loss: 1.5838344097137451, Val Accuracy: 46.0
Iteration 300, Epoch 1, Loss: 1.7288485765457153, Accuracy: 39.218231201171875, Val Loss: 1.5524072647094727, Val Accuracy: 47.70000076293945
Iteration 400, Epoch 1, Loss: 1.6683961153030396, Accuracy: 41.104270935058594, Val Loss: 1.5035769939422607, Val Accuracy: 47.79999923706055
Iteration 500, Epoch 1, Loss: 1.6268484592437744, Accuracy: 42.48066329956055, Val Loss: 1.4352067708969116, Val Accuracy: 50.599998474121094
Iteration 600, Epoch 1, Loss: 1.5988630056381226, Accuracy: 43.51861572265625, Val Loss: 1.4051826000213623, Val Accuracy: 51.400001525878906
Iteration 700, Epoch 1, Loss: 1.573

In [18]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

766/766 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 1.5719 - sparse_categorical_accuracy: 0.4478 - val_loss: 1.3148 - val_sparse_categorical_accuracy: 0.5260
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.3327 - sparse_categorical_accuracy: 0.5215


[1.3327070474624634, 0.5214999914169312]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [19]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [20]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.8586487770080566, Accuracy: 6.25, Val Loss: 3.0046780109405518, Val Accuracy: 13.40000057220459
Iteration 100, Epoch 1, Loss: 2.2183291912078857, Accuracy: 28.480815887451172, Val Loss: 1.9047770500183105, Val Accuracy: 37.20000076293945
Iteration 200, Epoch 1, Loss: 2.0680670738220215, Accuracy: 32.26834487915039, Val Loss: 1.8251838684082031, Val Accuracy: 40.70000076293945
Iteration 300, Epoch 1, Loss: 1.991567611694336, Accuracy: 34.073917388916016, Val Loss: 1.895099401473999, Val Accuracy: 35.0
Iteration 400, Epoch 1, Loss: 1.922864556312561, Accuracy: 35.925811767578125, Val Loss: 1.7210899591445923, Val Accuracy: 41.20000076293945
Iteration 500, Epoch 1, Loss: 1.876650333404541, Accuracy: 37.05401611328125, Val Loss: 1.6693127155303955, Val Accuracy: 42.5
Iteration 600, Epoch 1, Loss: 1.8460009098052979, Accuracy: 37.9809684753418, Val Loss: 1.7004082202911377, Val Accuracy: 42.89999771118164
Iteration 700, Epoch 1, Loss: 1.8200106620788574, Accura

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [22]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.dropout1 = tf.keras.layers.Dropout(0.2)

        self.conv3 = tf.keras.layers.Conv2D(64, (3, 3), padding='same', kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(64, (3, 3), padding='same', kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.dropout2 = tf.keras.layers.Dropout(0.3)

        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(128, activation='relu', kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.dropout3 = tf.keras.layers.Dropout(0.5)
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax', kernel_initializer=initializer)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = input_tensor
        
        x = tf.nn.relu(self.bn1(self.conv1(x), training=training))
        x = tf.nn.relu(self.bn2(self.conv2(x), training=training))
        x = self.pool1(x)
        x = self.dropout1(x, training=training)

        x = tf.nn.relu(self.bn3(self.conv3(x), training=training))
        x = tf.nn.relu(self.bn4(self.conv4(x), training=training))
        x = self.pool2(x)
        x = self.dropout2(x, training=training)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn5(x, training=training)
        x = self.dropout3(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.9047117233276367, Accuracy: 1.5625, Val Loss: 4.423189640045166, Val Accuracy: 11.399999618530273
Iteration 700, Epoch 1, Loss: 1.528186559677124, Accuracy: 46.79029846191406, Val Loss: 1.1492459774017334, Val Accuracy: 59.10000228881836
Iteration 1400, Epoch 2, Loss: 1.070986032485962, Accuracy: 62.55659484863281, Val Loss: 0.936437726020813, Val Accuracy: 66.9000015258789
Iteration 2100, Epoch 3, Loss: 0.9560918211936951, Accuracy: 66.82501983642578, Val Loss: 0.8191031813621521, Val Accuracy: 71.9000015258789
Iteration 2800, Epoch 4, Loss: 0.864322304725647, Accuracy: 70.1447525024414, Val Loss: 0.9576799273490906, Val Accuracy: 66.0
Iteration 3500, Epoch 5, Loss: 0.8041763305664062, Accuracy: 72.11455535888672, Val Loss: 0.713890552520752, Val Accuracy: 75.30000305175781
Iteration 4200, Epoch 6, Loss: 0.7592920064926147, Accuracy: 73.51751708984375, Val Loss: 0.7861160039901733, Val Accuracy: 74.29999542236328
Iteration 4900, Epoch 7, Loss: 0.711915373

Опишите все эксперименты, результаты. Сделайте выводы.

<span>Были протестированы различные программные интерфейсы TensorFlow для реализации нейронных сетей: Subclassing API, Sequential API и Functional API. Модели базовой трехслойной архитектуры, реализованные через Subclassing и Sequential подходы, показали практически идентичные результаты — около 51-52% точности на валидации всего за одну эпоху обучения. Это подтверждает, что выбор конкретного API в Keras влияет в первую очередь на удобство разработки и гибкость архитектуры, не изменяя математической сути обучения при одинаковых гиперпараметрах. Применение высокоуровневого метода model.fit также продемонстрировало преимущество в скорости работы и стабильности по сравнению с ручным циклом обучения, обеспечив итоговую точность на тесте на уровне 52.1%. </span>

<span>При использовании кастомной нейросети для достижения порога в 70% архитектура была значительно усложнена. Внедрены два блока сверток с увеличивающимся количеством фильтров, а также важные слои Batch Normalization и Dropout. Использование пакетной нормализации позволило стабилизировать распределение активаций, что в сочетании с оптимизатором Adam обеспечило стремительный рост точности: уже на третьей эпохе был преодолен порог в 70%, а к десятой эпохе модель достигла 78.1% точности на обучении и 76.9% на валидации.</span>

<span>Финальный анализ логов обучения показывает, что модель сохраняет отличную обобщающую способность. Разрыв между точностью на тренировочном и валидационном наборах минимален (около 1.2%), что свидетельствует об эффективности слоев Dropout в предотвращении переобучения, которое было заметно в предыдущей лабораторной работе. Снижение ошибки с 3.9 до 0.62 за 10 эпох подтверждает правильность выбранной стратегии инициализации весов и скорости обучения. Таким образом, использование фреймворка TensorFlow позволило спроектировать глубокую нейросеть, значительно превосходящую по качеству классификации все предыдущие подходы.</span>